In [1]:
!pip install -q sagemaker xgboost==1.7.6 imblearn scikit-learn s3fs pandas boto3 joblib

# Importações Padrão
import sagemaker
import boto3
import os
import pandas as pd
import time
import joblib # Para salvar/carregar o scaler
try:
    from sagemaker.xgboost import XGBoost
except ImportError:
    print("XGBoost ainda não instalado corretamente.")

print("Instalação/Importação tentada.")

/home/ec2-user/anaconda3/envs/tensorflow2_p310/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Instalação/Importação tentada.


In [2]:
# Inicializa a sessão SageMaker
sess = sagemaker.Session()

# Obtém a IAM Role associada a esta instância de notebook
try:
    role = sagemaker.get_execution_role()
except ValueError:
    # Se não conseguir obter automaticamente (raro em instâncias SageMaker),
    # você precisará colar o ARN da role aqui manualmente:
    # Ex: role = "arn:aws:iam::123456789012:role/YourSageMakerRoleName"
    print("Não foi possível obter a role automaticamente. Verifique as permissões ou insira o ARN manualmente.")
    role = "COLE_O_ARN_DA_SUA_ROLE_AQUI" # Substitua isto

# Bucket padrão do SageMaker (normalmente criado automaticamente)
default_bucket = sess.default_bucket()
region = sess.boto_region_name

# --- Configure seus caminhos S3 ---
# Bucket onde seus dados de entrada estão
data_bucket = 'ons-risk-prediction-data-674650987717'
# Pasta dentro do bucket onde estão os arquivos data_N_*.parquet
data_prefix = 'export'
s3_input_data_path = f's3://{data_bucket}/{data_prefix}/'

# Caminho no bucket padrão do SageMaker onde os modelos treinados serão salvos
s3_model_output_prefix = 'ons-risk-prediction/xgboost-models'
s3_model_output_path = f's3://{default_bucket}/{s3_model_output_prefix}/'
# --- Fim da Configuração ---

print(f"SageMaker Session: Initialized")
print(f"Execution Role ARN: {role}")
print(f"Default SageMaker Bucket: {default_bucket}")
print(f"AWS Region: {region}")
print(f"S3 Input Data Path: {s3_input_data_path}")
print(f"S3 Model Output Path: {s3_model_output_path}")

SageMaker Session: Initialized
Execution Role ARN: arn:aws:iam::674650987717:role/service-role/SageMaker-DataEngineer
Default SageMaker Bucket: sagemaker-us-east-1-674650987717
AWS Region: us-east-1
S3 Input Data Path: s3://ons-risk-prediction-data-674650987717/export/
S3 Model Output Path: s3://sagemaker-us-east-1-674650987717/ons-risk-prediction/xgboost-models/


In [3]:
source_dir = 'source_xgb'
os.makedirs(source_dir, exist_ok=True)
print(f"Diretório '{source_dir}' criado ou já existe.")
print("\nAVISO: As próximas duas células criarão o CONTEÚDO dos arquivos.")
print(f"Certifique-se de que os arquivos 'requirements.txt' e 'train_xgboost.py' sejam realmente salvos dentro da pasta '{source_dir}' no explorador de arquivos do JupyterLab antes de executar a célula final do 'fit'.")

Diretório 'source_xgb' criado ou já existe.

AVISO: As próximas duas células criarão o CONTEÚDO dos arquivos.
Certifique-se de que os arquivos 'requirements.txt' e 'train_xgboost.py' sejam realmente salvos dentro da pasta 'source_xgb' no explorador de arquivos do JupyterLab antes de executar a célula final do 'fit'.


In [4]:
%%writefile {source_dir}/requirements.txt
xgboost==1.7.6
imblearn
scikit-learn==0.24.1
pandas
joblib

Overwriting source_xgb/requirements.txt


In [5]:
%%writefile source_xgb/train_xgboost.py
import argparse
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import joblib
import sys
import traceback

# --- DEFINIÇÃO COMPLETA DOS GRUPOS DE FEATURES ---
features_carga = ['programada', 'verificada', 'diferenca_verif_prog']
features_geracao = ['geracao_total_diaria_go', 'geracao_fotovoltaica_diaria', 'geracao_hidroelétrica_diaria', 'geracao_térmica_diaria']
features_rede = ['total_mwh_restrito_go', 'saldo_intercambio_seco', 'restricao_razao_rel', 'restricao_razao_cnf', 'restricao_razao_ene', 'restricao_razao_par', 'restricao_origem_loc', 'restricao_origem_sis']
features_hidrica = ['ear_percentual_seco', 'ena_percentual_mlt_seco']
features_adicionais = ['cmo_semanal_seco', 'disponibilidade_total_diaria_go', 'indicador_ccal_mensal']
features_clima = ['ghi', 'temp2m_c', 'precipitacao_mm']
features_avancadas = ['carga_media_7d', 'carga_std_7d', 'geracao_media_7d', 'ear_ontem', 'ear_diff_3d', 'margem_oferta_demanda', 'pressao_demanda_ear', 'precip_acumulada_14d', 'precip_acumulada_30d']
features_calendario = ['mes', 'dia_da_semana', 'dia_do_ano']

def _get_features_for_scenario(cenario, all_columns):
    print(f"Selecionando features para o cenário: {cenario}")
    if cenario == 'pos_2022':
        features_para_teste_nomes = (features_geracao + features_rede + features_hidrica + features_adicionais + features_clima + features_calendario + features_avancadas + ['programada'])
    elif cenario == 'pos_2020':
        features_para_teste_nomes = (features_hidrica + features_adicionais + features_clima + features_calendario + features_avancadas + ['programada'])
    elif cenario == 'pos_2017' or cenario == 'pos_2015':
        features_para_teste_nomes = (features_hidrica + features_adicionais + features_clima + features_calendario + features_avancadas)
    elif cenario == 'pos_2013' or cenario == 'pos_2010':
        features_para_teste_nomes = (features_hidrica + features_clima + features_calendario + features_avancadas + ['cmo_semanal_seco', 'indicador_ccal_mensal'])
    else:
        raise ValueError(f"Cenário '{cenario}' não reconhecido no script.")

    selected_features = [col for col in features_para_teste_nomes if col in all_columns]
    print(f"Features selecionadas ({len(selected_features)}): {selected_features}")
    if not selected_features:
        raise ValueError("Nenhuma feature foi selecionada. Verifique o cenário e as colunas do dataframe.")
    return selected_features


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--model-dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN'))
    parser.add_argument('--n_estimators', type=int, default=1500)
    parser.add_argument('--max_depth', type=int, default=9)
    parser.add_argument('--learning_rate', type=float, default=0.05)
    parser.add_argument('--subsample', type=float, default=0.8)
    parser.add_argument('--colsample_bytree', type=float, default=0.7)
    parser.add_argument('--gamma', type=float, default=0)
    parser.add_argument('--reg_alpha', type=float, default=0)
    parser.add_argument('--reg_lambda', type=float, default=2)
    
    parser.add_argument('--objective', type=str, default='multi:softprob') 
    
    parser.add_argument('--num_class', type=int, default=3)
    parser.add_argument('--eval_metric', type=str, default='mlogloss')
    parser.add_argument('--cenario_escolhido', type=str, required=True)
    args = parser.parse_args()

    print("--- Iniciando Script de Treinamento XGBoost no SageMaker ---")
    print(f"Hiperparâmetros recebidos: {vars(args)}")
    print(f"Diretório de dados de treino: {args.train}")
    print(f"Diretório de saída do modelo: {args.model_dir}")

    try:
        # --- Carregar Dados ---
        input_files_path = args.train
        all_files = [os.path.join(input_files_path, f) for f in os.listdir(input_files_path) if f.endswith('.parquet')]
        if not all_files:
            raise ValueError(f"Nenhum arquivo .parquet encontrado em {input_files_path}")

        print(f"Lendo {len(all_files)} arquivo(s) parquet de {input_files_path}...")
        list_of_dfs = []
        for f in all_files:
            try:
                df_temp = pd.read_parquet(f)
                print(f"  Lido {f}, shape: {df_temp.shape}")
                list_of_dfs.append(df_temp)
            except Exception as read_err:
                print(f"!!!!!!!! ERRO AO LER O ARQUIVO {f}: {read_err} !!!!!!!!")
                print("        -> Pulando este arquivo.")

        if not list_of_dfs:
            raise ValueError("Nenhum DataFrame foi carregado com sucesso.")

        df_completo = pd.concat(list_of_dfs, ignore_index=True)
        print(f"Dados carregados e concatenados. Shape final: {df_completo.shape}")
        print(f"Colunas encontradas: {df_completo.columns.tolist()}")

        # --- Verificação e Seleção ---
        target_column_expected = 'nivel_risco'
        target_column_actual = None
        
        if target_column_expected in df_completo.columns:
            target_column_actual = target_column_expected
            print(f"Coluna target '{target_column_actual}' encontrada.")
            features_para_treino = _get_features_for_scenario(args.cenario_escolhido, df_completo.columns)
            X = df_completo[features_para_treino]
        elif '_COL_0' in df_completo.columns:
            print(f"AVISO: Usando colunas genéricas (_COL_X)")
            target_column_actual = df_completo.columns[-1]
            potential_feature_cols = df_completo.columns[:-1].tolist()
            X = df_completo[potential_feature_cols].select_dtypes(include=np.number)
            features_para_treino = X.columns.tolist()
            print(f"Usando {len(features_para_treino)} colunas numéricas como features")
            df_completo.rename(columns={target_column_actual: target_column_expected}, inplace=True)
            target_column_actual = target_column_expected
        else:
            raise ValueError(f"Coluna target '{target_column_expected}' não encontrada")

        y = df_completo[target_column_actual]

        # --- DEBUG TARGET ---
        print(f"\n--- DEBUG TARGET ---")
        print(f"Tipo: {y.dtype}")
        print(f"Valores únicos (amostra): {y.unique()[:20]}")
        print(f"Estatísticas:\n{y.describe()}")
        print(f"--- FIM DEBUG ---\n")

        # --- Pré-processamento do TARGET ---
        print("Processando target...")
        
        # Remover NaNs
        if y.isnull().any():
            print(f"Removendo {y.isnull().sum()} NaNs")
            valid_idx = y.dropna().index
            X = X.loc[valid_idx]
            y = y.loc[valid_idx]
        
        if y.dtype == 'object' or y.dtype.name == 'category':
            print("Target é categórico, mapeando para inteiros...")
            mapeamento_risco = {'baixo': 0, 'medio': 1, 'alto': 2}
            y_encoded = y.map(mapeamento_risco)
            if y_encoded.isnull().any():
                print(f"ERRO: {y_encoded.isnull().sum()} valores não mapeados!")
                print(f"Valores não mapeados: {y[y_encoded.isnull()].unique()}")
                raise ValueError("Valores categóricos não reconhecidos no target")
        else:
            # Target numérico contínuo: discretizar em 3 classes usando percentis
            print("Target numérico: discretizando em 3 classes (baixo, médio, alto)")
            print(f"Distribuição original - Min: {y.min():.2f}, Max: {y.max():.2f}, Média: {y.mean():.2f}")
            
            # Calcular percentis para dividir em 3 grupos equilibrados
            q1 = y.quantile(0.33)
            q2 = y.quantile(0.67)
            
            print(f"Percentil 33%: {q1:.2f}")
            print(f"Percentil 67%: {q2:.2f}")
            
            # Criar classes: 0=baixo, 1=médio, 2=alto
            y_encoded = pd.Series(index=y.index, dtype=int)
            y_encoded[y <= q1] = 0  # Baixo
            y_encoded[(y > q1) & (y <= q2)] = 1  # Médio
            y_encoded[y > q2] = 2  # Alto
            
            # Salvar os thresholds para uso posterior
            thresholds = {'q1': q1, 'q2': q2}
            thresholds_path = os.path.join(args.model_dir, 'risk_thresholds.joblib')
            joblib.dump(thresholds, thresholds_path)
            print(f"Thresholds salvos em: {thresholds_path}")
        
        y_encoded = y_encoded.astype(int)
        num_classes = 3  # FORÇAR 3 classes
        print(f"\nNúmero de classes: {num_classes}")
        print(f"Distribuição:\n{y_encoded.value_counts().sort_index()}")
        
        # IMPORTANTE: Atualizar num_class baseado nos dados reais
        if num_classes != args.num_class:
            print(f"AVISO: Ajustando num_class de {args.num_class} para {num_classes}")
            args.num_class = num_classes

        # --- Pré-processamento das FEATURES ---
        print("\nProcessando features...")
        
        # Verificar e remover colunas com variância zero ou todas NaN
        print(f"Features antes da limpeza: {X.shape[1]}")
        
        # Remover colunas que são totalmente NaN
        X = X.dropna(axis=1, how='all')
        print(f"Após remover colunas vazias: {X.shape[1]}")
        
        # Remover colunas com variância zero (todas iguais)
        variance = X.var()
        cols_to_keep = variance[variance > 0].index
        X = X[cols_to_keep]
        print(f"Após remover colunas com variância zero: {X.shape[1]}")
        
        # Preencher NaNs restantes com a mediana (mais robusto que média)
        if X.isnull().sum().any():
            print(f"Preenchendo {X.isnull().sum().sum()} NaNs com mediana")
            X = X.fillna(X.median())
            # Se ainda houver NaN (coluna toda NaN não foi dropada), preencher com 0
            X = X.fillna(0)
        
        # Substituir infinitos por valores finitos
        X = X.replace([np.inf, -np.inf], np.nan)
        if X.isnull().sum().any():
            print(f"Substituindo infinitos: {X.isnull().sum().sum()} valores")
            X = X.fillna(X.median()).fillna(0)
        
        print(f"Features finais: {X.shape[1]} colunas")
        print(f"Verificação final - NaNs: {X.isnull().sum().sum()}, Infs: {np.isinf(X).sum().sum()}")

        # Capturar a lista final de features (colunas de X) ANTES de escalar
        final_feature_names = X.columns.tolist()
        print(f"Capturando {len(final_feature_names)} nomes de features finais.")
        
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        print(f"Features escaladas. Shape: {X_scaled.shape}")
        
        # Verificação pós-scaling
        if np.isnan(X_scaled).any() or np.isinf(X_scaled).any():
            print("AVISO: Ainda há NaN/Inf após scaling. Aplicando limpeza final...")
            X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

        # --- SMOTE ---
        print("\nAplicando SMOTE...")
        min_count = y_encoded.value_counts().min()
        print(f"Classe minoritária: {min_count} amostras")
        
        if min_count > 5:
            k = min(5, min_count - 1)
            print(f"Usando k_neighbors={k}")
            smote = SMOTE(random_state=42, k_neighbors=k)
            X_resampled, y_resampled = smote.fit_resample(X_scaled, y_encoded)
            print(f"SMOTE aplicado. Shape: {X_resampled.shape}")
            print(f"Distribuição pós-SMOTE:\n{pd.Series(y_resampled).value_counts().sort_index()}")
        else:
            print(f"Pulando SMOTE (classe minoritária muito pequena)")
            X_resampled, y_resampled = X_scaled, y_encoded.values

        # --- Treinamento ---
        print("\nTreinando XGBClassifier...")
        model = xgb.XGBClassifier(
            n_estimators=args.n_estimators,
            max_depth=args.max_depth,
            learning_rate=args.learning_rate,
            subsample=args.subsample,
            colsample_bytree=args.colsample_bytree,
            gamma=args.gamma,
            reg_alpha=args.reg_alpha,
            reg_lambda=args.reg_lambda,
            objective=args.objective, # Agora será 'multi:softprob'
            num_class=args.num_class,
            eval_metric=args.eval_metric,
            random_state=42,
            use_label_encoder=False
        )
        
        model.fit(X_resampled, y_resampled)
        print("Treinamento concluído!")

        # --- Salvamento ---
        model_path = os.path.join(args.model_dir, 'xgboost-model.json')
        scaler_path = os.path.join(args.model_dir, 'scaler_xgb.joblib')
        
        print(f"\nSalvando modelo em: {model_path}")
        model.save_model(model_path)
        
        print(f"Salvando scaler em: {scaler_path}")
        joblib.dump(scaler, scaler_path)
        
        # Salvar a lista de features que foram usadas para treinar o scaler
        features_path = os.path.join(args.model_dir, 'feature_names_xgb.joblib')
        print(f"Salvando {len(final_feature_names)} features em: {features_path}")
        joblib.dump(final_feature_names, features_path)
        
        print("\n--- Script Finalizado com Sucesso ---")

    except Exception as e:
        print(f"\nERRO DURANTE O TREINAMENTO:")
        print(f"{str(e)}\n")
        traceback.print_exc()
        sys.exit(255)

Overwriting source_xgb/train_xgboost.py


In [6]:
# Cancelar job travado
import boto3

job_name_to_stop = 'xgb-ons-risk-pos-2010-20251019-173747'

try:
    sm_client = boto3.client('sagemaker', region_name=region)
    sm_client.stop_training_job(TrainingJobName=job_name_to_stop)
    print(f"✅ Job {job_name_to_stop} cancelado!")
except Exception as e:
    print(f"⚠️ Erro: {e}")

⚠️ Erro: An error occurred (ValidationException) when calling the StopTrainingJob operation: The request was rejected because the training job is in status Stopped.


In [7]:
# --- Configurar e Lançar o Job de Treinamento XGBoost ---

hyperparameters_xgb = {
    'n_estimators': 1500,
    'max_depth': 9,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'gamma': 0,
    'reg_alpha': 0,
    'reg_lambda': 2,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'cenario_escolhido': 'pos_2010'
}

# Cenário específico para nomear o job e artefatos
# *** MODIFICAÇÃO AQUI: Substituir _ por - ***
cenario_escolhido_tag = hyperparameters_xgb['cenario_escolhido'].replace('_', '-') # Ex: 'pos_2010' -> 'pos-2010'

# (Estimador e Input permanecem os mesmos...)
# Define o Estimator SageMaker XGBoost
xgb_estimator = XGBoost(
    entry_point='train_xgboost.py',
    source_dir=source_dir,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    framework_version='1.7-1',
    py_version='py3',
    hyperparameters=hyperparameters_xgb,
    output_path=s3_model_output_path,
    sagemaker_session=sess
)

# Define a localização dos dados de treinamento no S3
train_input_xgb = sagemaker.inputs.TrainingInput(
    s3_data=s3_input_data_path,
    distribution='FullyReplicated',
    content_type='parquet',
    s3_data_type='S3Prefix'
)

# --- Lança o Training Job ---
timestamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
job_name = f'xgb-ons-risk-{cenario_escolhido_tag}-{timestamp}' # Agora usa 'pos-2010'

print(f"Iniciando Job de Treinamento SageMaker: {job_name}")
print(f"Usando dados de: {s3_input_data_path}")
print(f"O modelo será salvo em S3 sob o prefixo: {s3_model_output_path}")
print(f"Hiperparâmetros: {hyperparameters_xgb}")

# Inicia o job. wait=True faz a célula aguardar a conclusão.
xgb_estimator.fit({'train': train_input_xgb}, job_name=job_name, wait=True)

# --- Recupera o caminho do modelo treinado ---
xgb_model_s3_uri = xgb_estimator.model_data
print(f"\nJob de Treinamento {job_name} concluído.")
print(f"Artefato do modelo (model.tar.gz contendo modelo e scaler) salvo em: {xgb_model_s3_uri}")

INFO:sagemaker:Creating training-job with name: xgb-ons-risk-pos-2010-20251021-000605


Iniciando Job de Treinamento SageMaker: xgb-ons-risk-pos-2010-20251021-000605
Usando dados de: s3://ons-risk-prediction-data-674650987717/export/
O modelo será salvo em S3 sob o prefixo: s3://sagemaker-us-east-1-674650987717/ons-risk-prediction/xgboost-models/
Hiperparâmetros: {'n_estimators': 1500, 'max_depth': 9, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.7, 'gamma': 0, 'reg_alpha': 0, 'reg_lambda': 2, 'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss', 'cenario_escolhido': 'pos_2010'}
2025-10-21 00:06:06 Starting - Starting the training job...
2025-10-21 00:06:39 Downloading - Downloading input data...
2025-10-21 00:07:04 Downloading - Downloading the training image......
2025-10-21 00:07:50 Training - Training image download completed. Training in progress./miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html